In [2]:
%pip install sweetviz

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

c:\Users\tyler\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [4]:
df = pd.read_csv('../datasets/insurance.csv')

df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [5]:
report = sv.analyze(df)
report.show_html('insurance_report.html')

                                             |          | [  0%]   00:00 -> (? left)

Report insurance_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [6]:
#encoded categorical variables
insurance_encoded = pd.get_dummies(df, drop_first=True).astype(int)
insurance_encoded.head()

,age,bmi,children,charges,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,19,27,0,16884,0,1,0,0,1
1,18,33,1,1725,1,0,0,1,0
2,28,33,3,4449,1,0,0,1,0
3,33,22,0,21984,1,0,1,0,0
4,32,28,0,3866,1,0,1,0,0


In [7]:
# prepare data for modeling
X = insurance_encoded.drop('charges', axis=1)
y = insurance_encoded['charges']

#split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#scale features (dont need for RF but do for linear regression)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#train linear regression model
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
#predict and evaluate linear regression
y_pred_lr = lr.predict(X_test_scaled)
mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

#train random forest model
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)
#predict and evaluate random forest
y_pred_rf = rf.predict(X_test)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

#print results
print(f'Linear Regression MSE: {mse_lr:.2f}, R2: {r2_lr:.2f}')
print(f'Random Forest MSE: {mse_rf:.2f}, R2: {r2_rf:.2f}')


Linear Regression MSE: 33566439.74, R2: 0.78
Random Forest MSE: 22179121.67, R2: 0.86


In [8]:
# CV with baseline Models with R2 as scoring metric

#CV for linear regression
cv_scores_lr = cross_val_score(lr, X_train_scaled, y_train, cv=5, scoring='r2')
print(f'Linear Regression CV R2 Scores: {cv_scores_lr}')

#CV for random forest
cv_scores_rf = cross_val_score(rf, X_train, y_train, cv=5, scoring='r2')
print(f'Random Forest CV R2 Scores: {cv_scores_rf}')

Linear Regression CV R2 Scores: [0.71508701 0.8018118  0.72333011 0.65748491 0.76763199]
Random Forest CV R2 Scores: [0.81530553 0.89851976 0.79006114 0.78263259 0.82825938]


In [9]:
#grid search for random forest
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'max_features': ['auto', 'sqrt', 'log2']
}

grid_search_rf = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search_rf.fit(X_train, y_train)

print(f'Best RF Parameters: {grid_search_rf.best_params_}')
print(f'Best RF CV R2 Score: {grid_search_rf.best_score_:.2f}')

c:\Users\tyler\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
60 fits failed out of a total of 180.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
46 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\tyler\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\tyler\anaconda3\Lib\site-packages\sklearn\base.py", line 1329, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\tyler\anaconda3\Lib\site-packages\sklearn\base.py", line 492, in _validate_params
   

Best RF Parameters: {'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 200}
Best RF CV R2 Score: 0.84


The best set of parameters is a max depth of 10, max features of log2, min samples split of 5, and number of estimators of 200. 

In [10]:
# examine top 20 ranked parameter combinations from grid search 
print("\nTop 20 GridSearchCV results:")
top20_eval = (
    pd.DataFrame(grid_search_rf.cv_results_)
    .sort_values('mean_test_score', ascending=False)
    .head(20)
    [['param_n_estimators', 'param_max_depth', 'param_min_samples_split', 
      'param_max_features', 'mean_test_score']]
    .rename(columns=lambda x: x.replace('param_', ''))
)
print(top20_eval)


Top 20 GridSearchCV results:
    n_estimators max_depth  min_samples_split max_features  mean_test_score
23           200        10                  5         log2         0.841202
11           200      None                  5         log2         0.840662
35           200        20                  5         log2         0.840662
22           100        10                  5         log2         0.840452
20           100        10                  2         log2         0.840383
21           200        10                  2         log2         0.839967
34           100        20                  5         log2         0.839195
10           100      None                  5         log2         0.839195
33           200        20                  2         log2         0.836639
9            200      None                  2         log2         0.836593
32           100        20                  2         log2         0.835721
8            100      None                  2         log2

I would chose this set of parameters n_estimators = 200, max_depth = 10,   min_samples_split= 5. max_features = log2 because it has the highest mean test score which means on average it is performing the best


In [11]:
# Evaluate tuned Random Forest on training and test sets
best_rf_params = grid_search_rf.best_params_
tuned_rf = RandomForestRegressor(random_state=42, **best_rf_params)
tuned_rf.fit(X_train, y_train)

y_train_pred = tuned_rf.predict(X_train)
y_test_pred = tuned_rf.predict(X_test)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

print(f'Tuned RF train R2: {train_r2:.3f}, test R2: {test_r2:.3f}')
print(f'Tuned RF train MSE: {train_mse:.2f}, test MSE: {test_mse:.2f}')

Tuned RF train R2: 0.930, test R2: 0.868
Tuned RF train MSE: 10067783.72, test MSE: 20430805.91


There seems to be some overfitting because the train R2 is a bit higher then the test R2 and the train MSE is much lower then the test MSE.